In [4]:
from connectors.spark_session import get_spark_session
from connectors.postgres_connector import PostgresConnector
from connectors.minio_connector import MinIOConnector


spark = get_spark_session()

postgres = PostgresConnector(spark)
minio = MinIOConnector(spark)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/spark/.ivy2/cache
The jars for the packages stored in: /home/spark/.ivy2/jars
org.postgresql#postgresql added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-881cb54c-0866-445b-be6e-649f23356f82;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/org/postgresql/postgresql/42.7.3/postgresql-42.7.3.jar ...
	[SUCCESSFUL ] org.postgresql#postgresql;42.7.3!postgresql.jar (527ms)
downloadi

In [5]:
tables = [
    "orders",
    "customers",
    "delivery_events",
    "drivers",
    "order_items",
    "payments",
    "refunds",
    "restaurants"
]

In [11]:
from pyspark.sql.functions import current_timestamp
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
def ingest_to_bronze(table_name):

    print(f"Ingesting {table_name}...")

    # Extract from Postgres
    df = postgres.read_query(f"select * from public.{table_name}")

    # Add ingestion metadata
    df = df.withColumn("ingestion_timestamp", current_timestamp())

    bronze_path = f"s3a://deliverylake/bronze/{table_name}"

    # Write to Delta Bronze (append)
    df.write \
        .format("delta") \
        .mode("append") \
        .save(bronze_path)

    # Register table if not exists
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS bronze.{table_name}
        USING DELTA
        LOCATION '{bronze_path}'
    """)
    spark.sql(f"""optimize bronze.{table_name}""")

    print(f"{table_name} loaded successfully")

In [12]:
for table in tables:
    ingest_to_bronze(table)

Ingesting orders...
query select * from public.orders
orders loaded successfully
Ingesting customers...
query select * from public.customers
customers loaded successfully
Ingesting delivery_events...
query select * from public.delivery_events


delivery_events loaded successfully
Ingesting drivers...
query select * from public.drivers
drivers loaded successfully
Ingesting order_items...
query select * from public.order_items
order_items loaded successfully
Ingesting payments...
query select * from public.payments
payments loaded successfully
Ingesting refunds...
query select * from public.refunds
refunds loaded successfully
Ingesting restaurants...
query select * from public.restaurants
restaurants loaded successfully
